# Phần 8 — Ablation ConvNeXt-Tiny
**2A202602810 · Nguyễn Thái Anh**

B03/seed0 là alias T00: macro-F1 validation **96,6701%**, best epoch 10/12.
Notebook chạy **T01–T07 tuần tự**, chọn hai yếu tố bằng validation rồi chạy T08.
Giữ seed 0, fold 0, 12 epoch, batch 64, ảnh 224, LR/warmup/cosine và AMP của B03.
Mỗi run khởi tạo lại model/optimizer/RNG; không tiếp tục train từ checkpoint B03.

Chọn kernel `.venv/bin/python` trên máy có GPU. `RUN_TRAINING=False` chỉ kiểm tra và đọc kết quả.
Không chạy master đồng thời với các notebook T riêng. Run All tự resume cùng config hoặc đọc run đã hoàn tất.
Baseline và final nhiều seed thuộc phần 10; chọn ở đây chỉ dùng validation một seed.

## 1. Môi trường

In [ ]:
import os, sys, json
from pathlib import Path
from dataclasses import asdict
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo")
CODE = ROOT / "submissions/2A202602810_NguyenThaiAnh/code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
import ablation_notebook as ab
import backbone_notebook as nbtrain
import train as trainer
assert Path(trainer.__file__).resolve() == CODE / "train.py"
print("ROOT:", ROOT)
print("Python:", sys.executable)

## 2. Phạm vi chạy
`RUN_IDS` có thể là một phần T01–T07; T00 không được train lại.
Để hoàn thành phần 8, giữ đủ T01–T07 và `RUN_T08=True`.
Mặc định chạy tuần tự để tránh cạnh tranh GPU; thời gian train không thay thế latency phần 9.

In [ ]:
RUN_TRAINING = True
RUN_IDS = [f"T{i:02}" for i in range(1, 8)]
RUN_T08 = True
MIN_DELTA = 0.002

## 3. Kiểm tra nền T00 và ma trận
T01 scratch, T02 frozen; T03 color, T04 CutMix; T05 label smoothing, T06 focal, T07 weighted CE.
Các tham số đi kèm loss/mix thuộc cùng một yếu tố. Weighted CE chỉ lấy tần suất từ train.
Baseline được kiểm tra đủ epoch, CSV/hash, metadata, logits và metric khớp evaluator.

In [ ]:
plan = ab.active_plan()
configs = ab.configs_for(plan)
base = configs["T00"]
summary, baseline_history, metrics = ab.verify_ablation(base, base)
print(f"T00 = B03/seed{base.seed}; F1 val = {metrics['macro_f1']:.4%}; đủ {base.epochs} epoch")
display(pd.DataFrame([{"exp_id":s.exp_id, "axis":s.axis, "changes":s.changes,
                       "status":nbtrain.training_status(configs[s.exp_id])} for s in plan]))
print("Không tái train B03; mọi run T kế thừa seed/split/batch/LR/scheduler/preprocessing của B03.")

## 4. Train T01–T07
Mỗi process có lock ID/seed, log riêng, best.pt/last.pt và learning curve.
Interrupt dừng process con; chạy lại giữ cấu hình và resume từ epoch hoàn tất gần nhất.
Nếu mới có config nhưng chưa có last.pt, ghi nhận run lỗi và xử lý thư mục run đó trước khi bắt đầu lại.
AMP có thể giảm scale khi overflow; theo dõi loss/skip trong history. Không dùng yêu cầu tuyệt đối zero skip.

In [ ]:
if RUN_TRAINING:
    frame = ab.run_ablation(run_ids=RUN_IDS)
else:
    frame = ab.collect_results(require_complete=False)
display(frame[["exp_id", "status", "macro_f1_val", "delta_macro_f1_pp"]])

## 5. Chọn và train T08 sau khi T01–T07 hoàn tất
Tự chọn hai yếu tố khác nhau có Δ cao nhất; không ghép scratch với frozen hoặc hai loss loại trừ nhau.
Khi cả hai vượt ngưỡng thực dụng, ghi `validation_improvements`; nếu thiếu hai yếu tố đạt ngưỡng,
ghi `exploratory` và thử tổ hợp thăm dò. Không gọi yếu tố không cải thiện là yếu tố thắng.
Quyết định/nguồn metric được khóa vào `t08_selection.json` trước khi train; chạy lại giữ nguyên.
T08 không mặc định là recipe cuối: vẫn có thể thua T00 hoặc từng thành phần.

In [ ]:
complete = set(frame.loc[frame["status"] == "complete", "exp_id"])
required = {f"T{i:02}" for i in range(8)}
decision = None
if required <= complete:
    saved = ab.ABLATION_OUTPUT / "t08_selection.json"
    decision = json.loads(saved.read_text()) if saved.exists() else ab.propose_combination(frame, min_delta=MIN_DELTA)
    display(decision)
    print("Ngưỡng thực dụng 0.002 = 0.2 điểm % F1; không phải kiểm định ý nghĩa thống kê.")
    if RUN_TRAINING and RUN_T08:
        decision = ab.lock_combination(decision)
        cfg08 = ab.configs_for(ab.active_plan())["T08"]
        nbtrain.run_in_subprocess(cfg08, planned_group=4)
        ab.verify_ablation(cfg08, base)
        print("T08 Output audit: PASS")
else:
    print("Hoàn thành các run trước T08:", sorted(required - complete))

## 6. Tổng hợp đầu ra và Δ đúng baseline

In [ ]:
frame = ab.collect_results(require_complete=False)
display(frame[["exp_id", "axis", "status", "macro_f1_val", "top1_val", "nll_val", "delta_macro_f1_pp"]])
print("Δ tính bằng điểm phần trăm; new/resume chưa có metric kết quả đầy đủ.")
print("Bảng:", ab.ABLATION_OUTPUT / "training_ablation.csv")

## 7. Macro-F1 và các lớp khó/hiếm
So sánh macro-F1, NLL, F1/recall từng lớp. Loss của các criterion khác nhau không so sánh trực tiếp.
Learning curve từng run nằm trong `submissions/.../curves/`; hình tổng hợp nằm trong `curves/ablation_convnext_tiny/`.

In [ ]:
fig = ab.plot_results(frame)
plt.show()
plt.close(fig)
class_table = pd.read_csv(ab.ABLATION_OUTPUT / "training_per_class_val.csv")
display(class_table.pivot(index="class_name", columns="exp_id", values=["f1", "recall"]))
print("Số ảnh từng lớp lấy từ toàn bộ validation; không thay bằng subset.")

## 8. Kết luận và điều kiện đi tiếp
Cần đủ T00–T08, ba trục (khởi tạo/augmentation/loss), log/curve, tổ hợp và nhận xét kết quả âm.
`screening_selection.json` lưu ứng viên cao nhất trên val, chưa khóa cấu hình chung kết.
Sau phần 8: triển khai suy luận/calibration/benchmark phần 9. Test chỉ đánh giá sau khi khóa phần 10.

In [ ]:
selection = json.loads((ab.ABLATION_OUTPUT / "screening_selection.json").read_text())
display(selection)
display(Markdown((ab.ABLATION_OUTPUT / "ablation_report.md").read_text()))
print("Phần 8:", "ĐỦ RUN; bổ sung nhận xét khoa học từ kết quả" if selection["section8_complete"] else "CHƯA ĐỦ RUN")